# Pagar LLM Gemma: few-shot retrieval Gemma-3-4B-it (VALID saja)

Sama persis dengan pagar_llm_v1 (Qwen), hanya model dasar diganti. Compute FP32 karena Gemma 3 rawan overflow di FP16 dan T4 tanpa BF16.

Rencana dari diskusi `2026-11-26-metode-terbaru-kualitas-terjemahan-jawa` (Ringkasan Akhir, metode 1).
Dua prompt: `zero` (instruksi tingkat saja) dan `few3` (3 contoh train asli, tingkat sama, satu per grup, cosine TF-IDF char_wb 3-5
sisi Indonesia, seri diputus index). NF4 + compute FP16, greedy, maksimal 96 token baru, chat template model.
Pilot 6 baris valid per tingkat per prompt (seed 42 dari index terurut); lanjut semua 175 baris hanya bila proyeksi x1,25 <= 1 GPU-jam.
Keputusan menang/gagal memakai audit fakta di luar notebook; chrF++ hanya pagar. Test tidak dibuka.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "sacrebleu", "scikit-learn"], check=False)
import glob, json, time, random, os
import numpy as np, pandas as pd, torch, transformers, sacrebleu
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.feature_extraction.text import TfidfVectorizer
_cfg = sorted(glob.glob("/kaggle/input/**/gemma*4b*it*/**/config.json", recursive=True) + glob.glob("/kaggle/input/**/gemma-3*/**/config.json", recursive=True))
print("isi input:", [p for p in glob.glob("/kaggle/input/**", recursive=True) if p.count("/") <= 7][:40])
assert _cfg, "Gemma tidak ter-mount: setujui lisensi Gemma di Kaggle"
MODEL, REV = os.path.dirname(_cfg[0]), None
print("model dir", MODEL, os.listdir(MODEL))
LEVELS = ["ngoko", "krama", "krama_alus"]
NAMA = {"ngoko": "Ngoko", "krama": "Krama (krama lugu)", "krama_alus": "Krama Alus (krama inggil)"}
OUT, PLAFON, MAXNEW, BS = "/kaggle/working", 3600, 96, 8
T0 = time.time(); log = lambda *a: print(f"[{time.time()-T0:6.0f}s]", *a, flush=True)
find = lambda n: sorted(p for p in glob.glob(f"/kaggle/input/**/{n}.csv", recursive=True) if "/output" not in p)[0]
def load3(n):
    d = pd.read_csv(find(n)); return d[d["level"].isin(LEVELS)].reset_index(drop=True)
tr, va = load3("train"), load3("valid")
assert (len(tr), len(va)) == (1395, 175), (len(tr), len(va))
MANIFEST = {"model": MODEL, "revision": REV, "versions": {"torch": torch.__version__, "transformers": transformers.__version__},
            "gpu": torch.cuda.get_device_name(0), "maxnew": MAXNEW, "greedy": True, "nf4": True, "compute": "fp32"}
log("data", len(tr), len(va), MANIFEST["gpu"])


In [ ]:
# ---------------------------------------------------- retrieval: 3 contoh train asli, tingkat sama, satu per grup
VEC = {l: TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit(tr[tr.level == l].indonesian) for l in LEVELS}
def contoh(ind, lv, k=3):
    T = tr[tr.level == lv]
    s = (VEC[lv].transform([ind]) @ VEC[lv].transform(T.indonesian).T).toarray()[0]
    order = sorted(range(len(T)), key=lambda i: (-round(float(s[i]), 12), int(T["index"].iloc[i])))
    out, seen = [], set()
    for i in order:
        g = T["group"].iloc[i]
        if g in seen:
            continue
        seen.add(g); out.append((T.indonesian.iloc[i], T.javanese.iloc[i], round(float(s[i]), 4)))
        if len(out) == k:
            break
    return out
SYS = "Anda penerjemah bahasa Indonesia ke bahasa Jawa yang menguasai unggah-ungguh (tingkat tutur)."
def instr(ind, lv):
    return (f"Terjemahkan kalimat bahasa Indonesia berikut ke bahasa Jawa tingkat tutur {NAMA[lv]}. "
            f"Pertahankan makna, pelaku, objek, dan negasi. Jawab hanya dengan satu kalimat terjemahan, tanpa penjelasan.\n\n{ind}")
def pesan(ind, lv, arm):
    m = [{"role": "system", "content": SYS}]
    if arm == "few3":
        for s, t, _ in contoh(ind, lv):
            m += [{"role": "user", "content": instr(s, lv)}, {"role": "assistant", "content": t}]
    m = m + [{"role": "user", "content": instr(ind, lv)}]
    return [{"role": x["role"], "content": [{"type": "text", "text": x["content"]}]} for x in m]


In [ ]:
# ---------------------------------------------------- model NF4
tok = AutoTokenizer.from_pretrained(MODEL); tok.padding_side = "left"
PAD = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float32)
t = time.time()
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=q, device_map="cuda:0", dtype=torch.float32).eval()
except Exception as e:
    print("CausalLM gagal, coba ImageTextToText:", repr(e)[:300])
    from transformers import AutoModelForImageTextToText
    model = AutoModelForImageTextToText.from_pretrained(MODEL, quantization_config=q, device_map="cuda:0", dtype=torch.float32).eval()
MANIFEST["load_s"] = round(time.time() - t); log("load", MANIFEST["load_s"], "s")
def bersih(x):
    x = x.strip().split("\n")[0].strip()
    return x.strip('"\u201c\u201d').strip()
def gen(rows, arm):
    out = []
    for i in range(0, len(rows), BS):
        b = rows[i:i + BS]
        txt = [tok.apply_chat_template(pesan(r.indonesian, r.level, arm), tokenize=False, add_generation_prompt=True) for r in b]
        enc = tok(txt, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            g = model.generate(**enc, max_new_tokens=MAXNEW, do_sample=False, temperature=None, top_p=None, top_k=None, pad_token_id=PAD)
        new = g[:, enc["input_ids"].shape[1]:]
        raw = tok.batch_decode(new, skip_special_tokens=True)
        out += [(bersih(x), x, int((y != PAD).sum())) for x, y in zip(raw, new)]
    return out


In [ ]:
# ---------------------------------------------------- pilot lalu penuh
srt = va.sort_values("index").reset_index(drop=True)
pilot_idx = sorted(sum([random.Random(42).sample(list(srt[srt.level == l]["index"]), 6) for l in LEVELS], []))
HAS = {}
torch.cuda.reset_peak_memory_stats(); t = time.time()
for arm in ["zero", "few3"]:
    rows = [r for r in srt.itertuples() if r.index in pilot_idx]
    HAS[arm] = dict(zip([r.index for r in rows], gen(rows, arm)))
pilot_s = time.time() - t
proyeksi = MANIFEST["load_s"] + pilot_s * len(srt) / len(pilot_idx)
MANIFEST["pilot"] = {"index": pilot_idx, "detik": round(pilot_s), "proyeksi_total_s": round(proyeksi),
                     "vram_puncak_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2)}
log("pilot", MANIFEST["pilot"])
lanjut = proyeksi * 1.25 <= PLAFON
MANIFEST["lanjut_penuh"] = lanjut
if lanjut:
    for arm in ["zero", "few3"]:
        rows = [r for r in srt.itertuples() if r.index not in HAS[arm]]
        t = time.time(); HAS[arm].update(dict(zip([r.index for r in rows], gen(rows, arm))))
        log(arm, "penuh", round(time.time() - t), "s")
MANIFEST["vram_puncak_gb"] = round(torch.cuda.max_memory_allocated() / 2**30, 2)
res = {}
for arm, h in HAS.items():
    d = srt[srt["index"].isin(h)].copy()
    d["hyp"] = [h[i][0] for i in d["index"]]; d["raw"] = [h[i][1] for i in d["index"]]; d["n_tok"] = [h[i][2] for i in d["index"]]
    if arm == "few3":
        d["contoh"] = [json.dumps(contoh(r.indonesian, r.level), ensure_ascii=False) for r in d.itertuples()]
    d.drop(columns=[c for c in ["label", "english"] if c in d]).to_csv(f"{OUT}/valid_{arm}.csv", index=False)
    res[arm] = {l: {"n": int((d.level == l).sum()), "chrF++": round(sacrebleu.corpus_chrf(d[d.level == l].hyp.tolist(), [d[d.level == l].javanese.tolist()], word_order=2).score, 2)}
                for l in LEVELS}
    res[arm]["kena_batas_token"] = int((d.n_tok >= MAXNEW).sum())
MANIFEST["chrF++"] = res; MANIFEST["elapsed_s"] = round(time.time() - T0)
json.dump(MANIFEST, open(f"{OUT}/manifest.json", "w"), indent=1, ensure_ascii=False)
print(json.dumps(res, indent=1)); log("SELESAI")
